# LeetCode 685 — Redundant Connection II bằng Python

- Link: https://leetcode.com/problems/redundant-connection-ii/
- Độ khó: **Hard**
- Chủ đề: Đồ thị có hướng, Cây có gốc, Union-Find (DSU), Phát hiện chu trình, Phân tích trường hợp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findRedundantDirectedConnection(self, edges: List[List[int]]) -> List[int]:

```

## 1. Giới thiệu & mục tiêu học

Đây là "người anh khó tính" của bài 684 *Redundant Connection*. Ở bài 684 (đồ thị vô hướng), ta chỉ cần tìm cạnh tạo chu trình bằng Union-Find. Ở bài này cạnh **có hướng**, và đồ thị ban đầu là một **cây có gốc** (mọi cạnh hướng từ cha xuống con). Thêm một cạnh thừa có thể phá cây theo **ba kiểu khác nhau**, và mẹo nằm ở chỗ nhận ra và xử lý đúng cả ba kiểu.

Sau notebook này bạn sẽ:

1. Nắm định nghĩa cây có gốc qua **bậc vào (in-degree)**: gốc có bậc vào 0, mọi đỉnh khác có bậc vào đúng 1, và mọi đỉnh đi tới được từ gốc.
2. Viết brute force "thử bỏ từng cạnh rồi kiểm tra có phải cây" làm trọng tài.
3. Phân loại 3 trường hợp: (a) một đỉnh có 2 cha, không chu trình; (b) một đỉnh có 2 cha và có chu trình; (c) không đỉnh nào có 2 cha nhưng có chu trình (cạnh thừa trỏ vào gốc).
4. Cài đặt Union-Find có nén đường đi và dùng nó cho đồ thị có hướng một cách cẩn thận.
5. Viết thêm lời giải "đi ngược con trỏ cha" và kiểm chứng mọi cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Một **cây có gốc** là đồ thị có hướng có đúng một đỉnh gốc, mọi đỉnh khác đều là hậu duệ của gốc, và mỗi đỉnh không phải gốc có **đúng một cha**.

Ban đầu có một cây có gốc với `n` đỉnh đánh số `1..n`. Người ta thêm **một cạnh có hướng** nối hai đỉnh khác nhau (cạnh này chưa có sẵn). Kết quả là danh sách `edges` gồm `n` cạnh, mỗi cạnh `[u, v]` nghĩa là `u` là cha của `v` (cạnh `u → v`).

Hãy trả về **một cạnh** mà khi bỏ đi thì đồ thị còn lại là cây có gốc với `n` đỉnh. Nếu có nhiều đáp án, trả về cạnh **xuất hiện sau cùng** trong `edges`.

**Ví dụ 1**

```text
Input:  edges = [[1,2],[1,3],[2,3]]
Output: [2,3]
Giải thích: đỉnh 3 có hai cha (1 và 2). Bỏ [1,3] hay [2,3] đều được cây; chọn cạnh xuất hiện sau: [2,3].
```

**Ví dụ 2**

```text
Input:  edges = [[1,2],[2,3],[3,4],[4,1],[1,5]]
Output: [4,1]
Giải thích: có chu trình 1→2→3→4→1, không đỉnh nào có hai cha. Phải bỏ cạnh [4,1] để 1 làm gốc.
```

**Ràng buộc**

- `n == edges.length`, `3 <= n <= 1000`
- `edges[i].length == 2`, `1 <= u_i, v_i <= n`, `u_i != v_i`

In [1]:
ex1 = [[1, 2], [1, 3], [2, 3]]
ex2 = [[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]]

def describe(edges):
    n = len(edges)
    indeg = [0] * (n + 1)
    for u, v in edges:
        indeg[v] += 1
    print("cạnh:", edges)
    print("bậc vào mỗi đỉnh:", {v: indeg[v] for v in range(1, n + 1)})

describe(ex1)
describe(ex2)

cạnh: [[1, 2], [1, 3], [2, 3]]
bậc vào mỗi đỉnh: {1: 0, 2: 1, 3: 2}
cạnh: [[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]]
bậc vào mỗi đỉnh: {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}


## 3. Kiến thức nền

### 3.1. Bậc vào (in-degree) và nhận diện cây có gốc

Với cạnh `u → v`, đỉnh `v` có thêm 1 bậc vào. Một đồ thị có hướng `n` đỉnh, `n - 1` cạnh là cây có gốc **khi và chỉ khi**:

1. đúng một đỉnh có bậc vào 0 (gốc), mọi đỉnh khác có bậc vào 1;
2. từ gốc đi tới được mọi đỉnh.

(Điều kiện 1 thôi chưa đủ: ví dụ `3 → 3`… không có vì `u != v`, nhưng có thể có chu trình rời gốc như `1 → 2`, `3 → 4`, `4 → 3` với gốc 1 — bậc vào đúng nhưng 3, 4 không tới được.)

In [2]:
from collections import deque

def is_rooted_tree(n, edges):
    """edges: list cạnh (u, v) có hướng, n đỉnh 1..n."""
    if len(edges) != n - 1:
        return False
    indeg = [0] * (n + 1)
    children = [[] for _ in range(n + 1)]
    for u, v in edges:
        indeg[v] += 1
        children[u].append(v)
    roots = [v for v in range(1, n + 1) if indeg[v] == 0]
    if len(roots) != 1 or any(indeg[v] > 1 for v in range(1, n + 1)):
        return False
    seen = {roots[0]}
    q = deque(roots)
    while q:
        u = q.popleft()
        for w in children[u]:
            if w not in seen:
                seen.add(w)
                q.append(w)
    return len(seen) == n

print(is_rooted_tree(3, [(1, 2), (1, 3)]))           # True
print(is_rooted_tree(4, [(1, 2), (3, 4), (4, 3)]))   # False: chu trình 3<->4 không tới được
print(is_rooted_tree(3, [(1, 3), (2, 3)]))           # False: 3 có hai cha

True
False
False


### 3.2. Union-Find (Disjoint Set Union — DSU)

DSU quản lý các **tập rời nhau**. Hai thao tác:

- `find(x)`: trả về "đại diện" của tập chứa `x`.
- `union(a, b)`: gộp hai tập.

Nếu khi thêm cạnh `(a, b)` mà `find(a) == find(b)` → `a`, `b` đã liên thông → cạnh này tạo **chu trình (vô hướng)**.

**Nén đường đi** (`parent[x] = find(parent[x])`) giúp các lần `find` sau gần như `O(1)`.

In [3]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n + 1))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # nén đường đi (nửa đường)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False          # đã cùng tập -> cạnh tạo chu trình
        self.parent[rb] = ra
        return True

d = DSU(5)
print(d.union(1, 2), d.union(2, 3), d.union(3, 1))   # True True False (cạnh 3-1 khép chu trình)
print([d.find(i) for i in range(1, 6)])

True True False
[1, 1, 1, 4, 5]


### 3.3. Mảng "cha" (`parent_of`) và đi ngược lên gốc

Trong cây có gốc, mỗi đỉnh có tối đa một cha → lưu `parent_of[v] = u`. Từ một đỉnh, đi `v → parent_of[v] → …` sẽ tới gốc. Nếu đi mãi mà quay về chính mình thì có **chu trình**.

In [4]:
parent_of = {2: 1, 3: 2, 4: 3, 1: 4}        # chu trình 1->2->3->4->1 (viết theo cha)
x, steps = 1, []
for _ in range(5):
    steps.append(x)
    x = parent_of.get(x, 0)
print("đi ngược từ 1:", steps)

đi ngược từ 1: [1, 4, 3, 2, 1]


### 3.4. So sánh list trong Python

`[1, 2] == [1, 2]` là `True` (so từng phần tử). Ta sẽ dùng điều này để bỏ qua đúng một cạnh ứng viên khi duyệt. Lưu ý đề đảm bảo cạnh thêm vào **chưa có sẵn**, nên không có hai cạnh trùng nhau.

In [5]:
print([1, 2] == [1, 2], [1, 2] == [2, 1], (1, 2) == [1, 2])

True False False


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Cạnh thừa `u → v` phá cây theo những kiểu nào?

Cây ban đầu: mọi đỉnh trừ gốc có đúng một cha. Thêm cạnh `u → v`:

- **Nếu `v` không phải gốc** → `v` giờ có **hai cha**.
  - (a) Nếu `u` **không** là hậu duệ của `v` → không có chu trình. Bỏ cạnh nào trong hai cạnh vào `v` cũng được → chọn cạnh **xuất hiện sau**.
  - (b) Nếu `u` là hậu duệ của `v` → có chu trình `v → … → u → v`. Phải bỏ cạnh vào `v` **nằm trên chu trình** (là `u → v`), bỏ cạnh còn lại thì chu trình vẫn còn.
- **Nếu `v` là gốc** → (c) gốc có cha, không đỉnh nào có hai cha, nhưng có chu trình đi qua gốc. Bỏ bất kỳ cạnh nào trên chu trình đều được (đỉnh cuối của cạnh bị bỏ trở thành gốc) → chọn cạnh trên chu trình **xuất hiện sau cùng**.

### 4.2. Tính tay ba ví dụ

| Ví dụ | Đỉnh 2 cha? | Chu trình? | Trường hợp | Đáp án |
|---|---|---|---|---|
| `[[1,2],[1,3],[2,3]]` | 3 (cha 1, 2) | không | (a) | cạnh sau: `[2,3]` |
| `[[2,1],[3,1],[4,2],[1,4]]` | 1 (cha 2, 3) | `1→4→2→1` | (b) | cạnh vào 1 trên chu trình: `[2,1]` |
| `[[1,2],[2,3],[3,4],[4,1],[1,5]]` | không | `1→2→3→4→1` | (c) | cạnh chu trình sau cùng: `[4,1]` |

Ở ví dụ giữa, nếu chọn "cạnh sau" như (a) ta sẽ trả `[3,1]` — **sai**, vì bỏ `[3,1]` thì chu trình `1→4→2→1` vẫn còn.

In [6]:
ex3 = [[2, 1], [3, 1], [4, 2], [1, 4]]
for e in ex3:
    rest = [tuple(x) for x in ex3 if x != e]
    print("bỏ", e, "-> cây có gốc?", is_rooted_tree(4, rest))

bỏ [2, 1] -> cây có gốc? True
bỏ [3, 1] -> cây có gốc? False
bỏ [4, 2] -> cây có gốc? False
bỏ [1, 4] -> cây có gốc? False


## 5. Cách 1 — Brute Force: thử bỏ từng cạnh từ cuối lên

**Ý tưởng.** Vì phải trả về cạnh **xuất hiện sau cùng** trong các đáp án, ta thử bỏ cạnh theo thứ tự **từ cuối về đầu**; cạnh đầu tiên mà phần còn lại là cây có gốc chính là đáp án.

In [7]:
def redundant_brute(edges):
    n = len(edges)
    for idx in range(n - 1, -1, -1):
        rest = [tuple(e) for j, e in enumerate(edges) if j != idx]
        if is_rooted_tree(n, rest):
            return list(edges[idx])
    return None          # theo đề không xảy ra

print(redundant_brute(ex1))   # [2, 3]
print(redundant_brute(ex2))   # [4, 1]
print(redundant_brute(ex3))   # [2, 1]

[2, 3]
[4, 1]
[2, 1]


**Độ phức tạp.** `n` lần thử × kiểm tra cây `O(n)` = `O(n²)`. Với `n ≤ 1000` là khoảng `10^6` thao tác — thực ra **vẫn qua được** trên LeetCode! Nhưng đây là bài học tốt để hiểu cấu trúc, và lời giải `O(n·α(n))` gần tuyến tính mới là thứ người phỏng vấn muốn.

**Nút thắt.** Ta thử cả những cạnh "hiển nhiên vô can". Thực tế, theo mục 4.1, đáp án luôn thuộc một tập ứng viên rất nhỏ: một trong hai cạnh vào đỉnh có hai cha, hoặc một cạnh trên chu trình.

## 6. Bước đột phá

**Insight:** Chỉ có hai "triệu chứng" có thể xuất hiện: **đỉnh hai cha** và **chu trình**. Tìm đỉnh hai cha rất dễ (đếm cha). Sau đó dùng mẹo:

> Tạm **bỏ cạnh ứng viên thứ hai** `cand2` (cạnh vào đỉnh hai cha xuất hiện sau). Rồi kiểm tra phần còn lại có chu trình không.

Quy tắc "nếu… thì…":

- **Nếu** có đỉnh hai cha (`cand1` = cạnh vào xuất hiện trước, `cand2` = cạnh vào xuất hiện sau):
  - **nếu** sau khi bỏ `cand2` mà **không còn chu trình** → trả `cand2` (trường hợp (a), hoặc (b) mà `cand2` chính là cạnh trên chu trình);
  - **nếu** vẫn còn chu trình → chu trình đi qua `cand1` → trả `cand1`.
- **Nếu** không có đỉnh hai cha → trường hợp (c): trả về **cạnh khép chu trình** — khi thêm cạnh theo thứ tự đề, cạnh đầu tiên làm hai đầu đã liên thông chính là cạnh trên chu trình xuất hiện **sau cùng**.

**Bất biến khi chạy DSU:** sau khi xử lý `i` cạnh đầu (bỏ qua `cand2`), các tập của DSU đúng bằng các thành phần liên thông (vô hướng) của các cạnh đó; nếu chưa có cạnh nào khép chu trình thì các cạnh đã thêm tạo thành một **rừng**.

**Vì sao có thể dùng DSU (vô hướng) cho đồ thị có hướng ở đây?** Sau khi đã bỏ `cand2`, mỗi đỉnh có **tối đa một cha**. Trong đồ thị như vậy, một chu trình vô hướng **bắt buộc** là chu trình có hướng (đi ngược con trỏ cha). Vậy phát hiện chu trình vô hướng là đủ.

## 7. Các cách tiếp theo

Tổng cộng 3 cách:

| Cách | Ý tưởng |
|---|---|
| 1 | Brute force bỏ từng cạnh từ cuối, kiểm tra cây |
| 2 | Tìm đỉnh hai cha → `cand1`, `cand2`; DSU bỏ qua `cand2` để phát hiện chu trình |
| 3 | Như Cách 2 nhưng phát hiện chu trình bằng **đi ngược con trỏ cha** (không cần DSU) |

### 7.1. Cách 2 — Phân tích trường hợp + Union-Find

**Giải thích.**

1. Duyệt cạnh, ghi `parent_of[v]`. Nếu `v` đã có cha → `cand1 = [parent_of[v], v]`, `cand2 = [u, v]` (không ghi đè `parent_of`).
2. Duyệt lại cạnh với DSU, **bỏ qua `cand2`**. Nếu gặp cạnh `[u, v]` mà `find(u) == find(v)`:
   - có `cand1` → trả `cand1`;
   - không có → trả `[u, v]`.
3. Không gặp chu trình → trả `cand2`.

In [8]:
def redundant_dsu(edges):
    n = len(edges)
    parent_of = [0] * (n + 1)
    cand1 = cand2 = None
    for u, v in edges:
        if parent_of[v]:
            cand1 = [parent_of[v], v]
            cand2 = [u, v]
        else:
            parent_of[v] = u
    dsu = DSU(n)
    for u, v in edges:
        if cand2 is not None and [u, v] == cand2:
            continue                                # tạm bỏ cand2
        if not dsu.union(u, v):                     # u, v đã liên thông -> chu trình
            return cand1 if cand1 is not None else [u, v]
    return cand2

for e in [ex1, ex2, ex3]:
    print(e, "->", redundant_dsu(e))

[[1, 2], [1, 3], [2, 3]] -> [2, 3]
[[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]] -> [4, 1]
[[2, 1], [3, 1], [4, 2], [1, 4]] -> [2, 1]


Giải thích từng dòng quan trọng:

- `if parent_of[v]:` — `v` đã có cha (giá trị khác 0) → đây là đỉnh hai cha. Vì chỉ có **một** cạnh thừa nên nhiều nhất một đỉnh có hai cha.
- Không gán `parent_of[v] = u` cho cạnh thứ hai → `parent_of` mô tả đồ thị **đã bỏ `cand2`**.
- `[u, v] == cand2` — bỏ qua đúng cạnh đó khi chạy DSU.
- `return cand1 if cand1 ... else [u, v]` — gộp hai trường hợp (b) và (c) trong một dòng.
- `return cand2` — trường hợp (a) (hoặc (b) với `cand2` trên chu trình).

**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def findRedundantDirectedConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        parent_of = [0] * (n + 1)
        cand1 = cand2 = None
        for u, v in edges:
            if parent_of[v]:
                cand1, cand2 = [parent_of[v], v], [u, v]
            else:
                parent_of[v] = u
        root = list(range(n + 1))
        def find(x):
            while root[x] != x:
                root[x] = root[root[x]]
                x = root[x]
            return x
        for u, v in edges:
            if [u, v] == cand2:
                continue
            ru, rv = find(u), find(v)
            if ru == rv:
                return cand1 if cand1 else [u, v]
            root[rv] = ru
        return cand2
```

**Độ phức tạp.** `O(n · α(n))` ≈ `O(n)` thời gian (α là hàm Ackermann ngược, ≤ 4 trong thực tế), `O(n)` bộ nhớ.

### 7.2. Cách 3 — Phân tích trường hợp + đi ngược con trỏ cha

**Giải thích.** Sau bước 1 của Cách 2, `parent_of` mô tả đồ thị đã bỏ `cand2`, mỗi đỉnh tối đa một cha.

- **Có đỉnh hai cha `v`.** Theo mục 4.1, nếu bỏ `cand2` mà còn chu trình thì chu trình **đi qua `v`** (nó phải dùng `cand1`). Vậy chỉ cần đi ngược từ `v` tối đa `n` bước: quay lại `v` → còn chu trình → trả `cand1`; ngược lại trả `cand2`.
- **Không có đỉnh hai cha.** Mọi đỉnh có đúng một cha → đồ thị "hàm" có đúng một chu trình. Đi ngược `n` bước từ đỉnh 1 chắc chắn đã rơi vào chu trình; từ đó đi tiếp một vòng để thu thập các đỉnh trên chu trình. Đáp án là cạnh `[parent_of[w], w]` (với `w` trên chu trình) xuất hiện **sau cùng** trong `edges`.

In [9]:
def redundant_walk(edges):
    n = len(edges)
    parent_of = [0] * (n + 1)
    cand1 = cand2 = None
    for u, v in edges:
        if parent_of[v]:
            cand1, cand2 = [parent_of[v], v], [u, v]
        else:
            parent_of[v] = u

    if cand1 is not None:
        v = cand1[1]
        x = parent_of[v]
        for _ in range(n):
            if x == 0:                 # tới gốc (không có cha) -> không chu trình
                break
            if x == v:                 # quay về v -> còn chu trình
                return cand1
            x = parent_of[x]
        return cand2

    # Không có đỉnh hai cha: tìm chu trình duy nhất
    x = 1
    for _ in range(n):
        x = parent_of[x]               # sau n bước chắc chắn đang ở trên chu trình
    on_cycle = {x}
    y = parent_of[x]
    while y != x:
        on_cycle.add(y)
        y = parent_of[y]
    for u, v in reversed(edges):       # cạnh trên chu trình xuất hiện sau cùng
        if v in on_cycle and parent_of[v] == u:
            return [u, v]

for e in [ex1, ex2, ex3]:
    print(e, "->", redundant_walk(e))

[[1, 2], [1, 3], [2, 3]] -> [2, 3]
[[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]] -> [4, 1]
[[2, 1], [3, 1], [4, 2], [1, 4]] -> [2, 1]


Lưu ý điều kiện `v in on_cycle and parent_of[v] == u`: cạnh `u → v` nằm trên chu trình khi và chỉ khi `v` trên chu trình **và** `u` là cha của `v` (trong trường hợp không có đỉnh hai cha, `parent_of[v]` luôn là cha duy nhất, nên điều kiện thứ hai luôn đúng với cạnh vào `v` — viết ra cho rõ ý).

**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def findRedundantDirectedConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        par = [0] * (n + 1)
        cand1 = cand2 = None
        for u, v in edges:
            if par[v]:
                cand1, cand2 = [par[v], v], [u, v]
            else:
                par[v] = u
        if cand1:
            v, x = cand1[1], par[cand1[1]]
            for _ in range(n):
                if x == 0: break
                if x == v: return cand1
                x = par[x]
            return cand2
        x = 1
        for _ in range(n):
            x = par[x]
        cyc, y = {x}, par[x]
        while y != x:
            cyc.add(y); y = par[y]
        for u, v in reversed(edges):
            if v in cyc:
                return [u, v]
```

**Độ phức tạp.** `O(n)` thời gian, `O(n)` bộ nhớ — không cần DSU.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái cho `ex3 = [[2,1],[3,1],[4,2],[1,4]]` (Cách 2)

**Bước 1 — tìm đỉnh hai cha:**

| Cạnh | `parent_of[v]` trước | Hành động |
|---|---|---|
| `[2,1]` | 0 | `parent_of[1] = 2` |
| `[3,1]` | 2 | đỉnh 1 hai cha: `cand1=[2,1]`, `cand2=[3,1]` |
| `[4,2]` | 0 | `parent_of[2] = 4` |
| `[1,4]` | 0 | `parent_of[4] = 1` |

**Bước 2 — DSU, bỏ qua `[3,1]`:**

| Cạnh | `find(u)`, `find(v)` | Kết quả |
|---|---|---|
| `[2,1]` | 2, 1 | gộp {1,2} |
| `[3,1]` | — | bỏ qua (cand2) |
| `[4,2]` | 4, rep{1,2} | gộp {1,2,4} |
| `[1,4]` | cùng đại diện | **chu trình** → có `cand1` → trả `[2,1]` |

### 8.2. `trace_dsu`

In [10]:
def trace_dsu(edges):
    n = len(edges)
    print("edges =", edges)
    parent_of = [0] * (n + 1)
    cand1 = cand2 = None
    for u, v in edges:
        if parent_of[v]:
            cand1, cand2 = [parent_of[v], v], [u, v]
            print(f"  [{u},{v}]: đỉnh {v} đã có cha {parent_of[v]} -> cand1={cand1}, cand2={cand2}")
        else:
            parent_of[v] = u
            print(f"  [{u},{v}]: parent_of[{v}] = {u}")
    dsu = DSU(n)
    for u, v in edges:
        if cand2 is not None and [u, v] == cand2:
            print(f"  DSU: bỏ qua cand2 {cand2}")
            continue
        ru, rv = dsu.find(u), dsu.find(v)
        if ru == rv:
            ans = cand1 if cand1 is not None else [u, v]
            print(f"  DSU: [{u},{v}] find={ru},{rv} -> CHU TRÌNH -> trả {ans}")
            return ans
        dsu.union(u, v)
        groups = {}
        for x in range(1, n + 1):
            groups.setdefault(dsu.find(x), []).append(x)
        print(f"  DSU: [{u},{v}] gộp -> các tập {sorted(groups.values())}")
    print(f"  Không có chu trình -> trả cand2 {cand2}")
    return cand2

for e in [ex1, ex2, ex3]:
    trace_dsu(e)
    print()

edges = [[1, 2], [1, 3], [2, 3]]
  [1,2]: parent_of[2] = 1
  [1,3]: parent_of[3] = 1
  [2,3]: đỉnh 3 đã có cha 1 -> cand1=[1, 3], cand2=[2, 3]
  DSU: [1,2] gộp -> các tập [[1, 2], [3]]
  DSU: [1,3] gộp -> các tập [[1, 2, 3]]
  DSU: bỏ qua cand2 [2, 3]
  Không có chu trình -> trả cand2 [2, 3]

edges = [[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]]
  [1,2]: parent_of[2] = 1
  [2,3]: parent_of[3] = 2
  [3,4]: parent_of[4] = 3
  [4,1]: parent_of[1] = 4
  [1,5]: parent_of[5] = 1
  DSU: [1,2] gộp -> các tập [[1, 2], [3], [4], [5]]
  DSU: [2,3] gộp -> các tập [[1, 2, 3], [4], [5]]
  DSU: [3,4] gộp -> các tập [[1, 2, 3, 4], [5]]
  DSU: [4,1] find=1,1 -> CHU TRÌNH -> trả [4, 1]

edges = [[2, 1], [3, 1], [4, 2], [1, 4]]
  [2,1]: parent_of[1] = 2
  [3,1]: đỉnh 1 đã có cha 2 -> cand1=[2, 1], cand2=[3, 1]
  [4,2]: parent_of[2] = 4
  [1,4]: parent_of[4] = 1
  DSU: [2,1] gộp -> các tập [[1, 2], [3], [4]]
  DSU: bỏ qua cand2 [3, 1]
  DSU: [4,2] gộp -> các tập [[1, 2, 4], [3]]
  DSU: [1,4] find=4,4 -> CHU TR

### 8.3. `trace_walk` — đi ngược con trỏ cha

In [11]:
def trace_walk(edges):
    n = len(edges)
    par = [0] * (n + 1)
    cand1 = cand2 = None
    for u, v in edges:
        if par[v]:
            cand1, cand2 = [par[v], v], [u, v]
        else:
            par[v] = u
    print("edges =", edges, "| parent_of (đã bỏ cand2) =", {v: par[v] for v in range(1, n + 1)})
    if cand1:
        v = cand1[1]
        path, x = [v], par[v]
        while x and x != v and len(path) <= n:
            path.append(x)
            x = par[x]
        path.append(x)
        verdict = "quay về v -> còn chu trình -> cand1" if x == v else "tới gốc (0) -> không chu trình -> cand2"
        print(f"  đỉnh hai cha {v}; đi ngược: {' -> '.join(map(str, path))}; {verdict}")
        return cand1 if x == v else cand2
    x = 1
    for _ in range(n):
        x = par[x]
    cyc, y = [x], par[x]
    while y != x:
        cyc.append(y)
        y = par[y]
    print(f"  không có đỉnh hai cha; chu trình (theo cha): {cyc}")
    for u, v in reversed(edges):
        if v in cyc:
            print(f"  cạnh trên chu trình xuất hiện sau cùng: [{u},{v}]")
            return [u, v]

for e in [ex1, ex2, ex3]:
    trace_walk(e)

edges = [[1, 2], [1, 3], [2, 3]] | parent_of (đã bỏ cand2) = {1: 0, 2: 1, 3: 1}
  đỉnh hai cha 3; đi ngược: 3 -> 1 -> 0; tới gốc (0) -> không chu trình -> cand2
edges = [[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]] | parent_of (đã bỏ cand2) = {1: 4, 2: 1, 3: 2, 4: 3, 5: 1}
  không có đỉnh hai cha; chu trình (theo cha): [4, 3, 2, 1]
  cạnh trên chu trình xuất hiện sau cùng: [4,1]
edges = [[2, 1], [3, 1], [4, 2], [1, 4]] | parent_of (đã bỏ cand2) = {1: 2, 2: 4, 3: 0, 4: 1}
  đỉnh hai cha 1; đi ngược: 1 -> 2 -> 4 -> 1; quay về v -> còn chu trình -> cand1


## 9. Code nộp LeetCode

Chọn **Cách 2** (phân tích trường hợp + Union-Find) — lời giải chuẩn, ngắn, gộp gọn các trường hợp.

In [12]:
from typing import List

class Solution:
    def findRedundantDirectedConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        parent_of = [0] * (n + 1)
        cand1 = cand2 = None
        for u, v in edges:
            if parent_of[v]:
                cand1, cand2 = [parent_of[v], v], [u, v]
            else:
                parent_of[v] = u

        root = list(range(n + 1))

        def find(x):
            while root[x] != x:
                root[x] = root[root[x]]
                x = root[x]
            return x

        for u, v in edges:
            if [u, v] == cand2:
                continue
            ru, rv = find(u), find(v)
            if ru == rv:
                return cand1 if cand1 else [u, v]
            root[rv] = ru
        return cand2

assert Solution().findRedundantDirectedConnection([[1, 2], [1, 3], [2, 3]]) == [2, 3]
assert Solution().findRedundantDirectedConnection([[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]]) == [4, 1]
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi cạnh thừa là `e* = u → v`.

**Trường hợp có đỉnh hai cha `v`.** Hai cạnh vào `v` là `cand1` (trước) và `cand2` (sau); `e*` là một trong hai. Mọi đáp án phải là một trong hai cạnh này (bỏ cạnh khác thì `v` vẫn hai cha).

- Nếu đồ thị bỏ `cand2` **không có chu trình**: nó có `n - 1` cạnh, mỗi đỉnh tối đa một cha, không chu trình → là rừng các cây có hướng; với `n - 1` cạnh trên `n` đỉnh, rừng không chu trình có đúng một thành phần → là cây có gốc. Vậy `cand2` là đáp án hợp lệ, và vì nó xuất hiện sau nên ưu tiên → trả `cand2` đúng.
- Nếu bỏ `cand2` **còn chu trình**: bỏ `cand2` không cho cây → đáp án duy nhất là `cand1`. (Đề đảm bảo có đáp án.)
- DSU phát hiện đúng sự tồn tại chu trình vì khi mỗi đỉnh tối đa một cha, chu trình vô hướng ⇔ chu trình có hướng: đi dọc một chu trình vô hướng, mỗi đỉnh chỉ có một cạnh vào nên các cạnh phải cùng chiều.

**Trường hợp không có đỉnh hai cha.** Mọi đỉnh có đúng một cha (n cạnh, n đỉnh, bậc vào mỗi đỉnh ≤ 1 ⇒ đúng 1). Đồ thị có đúng một chu trình (đi qua gốc cũ). Bỏ cạnh `w' → w` trên chu trình → `w` thành gốc, mọi đỉnh tới được từ `w` → cây. Bỏ cạnh ngoài chu trình → còn chu trình → không phải cây. Vậy đáp án là cạnh chu trình xuất hiện **sau cùng**. Khi thêm cạnh theo thứ tự vào DSU, các cạnh trước khi chu trình hoàn tất tạo thành rừng; cạnh đầu tiên khép chu trình chính là cạnh chu trình có chỉ số lớn nhất → DSU trả đúng.

**Cách 3** kiểm tra cùng các điều kiện bằng cách đi ngược con trỏ cha (mục 7.2).

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(n)` | `n` lần BFS kiểm tra cây |
| 2. Trường hợp + DSU | `O(n · α(n))` | `O(n)` | chuẩn, dễ nhớ |
| 3. Trường hợp + đi ngược cha | `O(n)` | `O(n)` | không cần DSU, nhưng phải xử lý chu trình (c) riêng |

Đo thời gian với `n = 1000` (cây ngẫu nhiên + một cạnh thừa):

In [13]:
import random, time

def random_case(n, rng):
    """Sinh cây có gốc ngẫu nhiên n đỉnh + 1 cạnh thừa hợp lệ, xáo trộn thứ tự cạnh."""
    order = list(range(1, n + 1))
    rng.shuffle(order)
    edges = []
    for i in range(1, n):
        child = order[i]
        par = order[rng.randrange(i)]
        edges.append([par, child])
    existing = {tuple(e) for e in edges}
    while True:
        u, v = rng.randint(1, n), rng.randint(1, n)
        if u != v and (u, v) not in existing:
            break
    edges.append([u, v])
    rng.shuffle(edges)
    return edges

rng = random.Random(685)
big = random_case(1000, rng)
for name, fn in [("brute", redundant_brute), ("dsu", redundant_dsu), ("walk", redundant_walk)]:
    t0 = time.perf_counter()
    r = fn(big)
    print(f"{name:<6}: {r}  {time.perf_counter() - t0:.4f}s")

brute : [820, 524]  0.0957s
dsu   : [820, 524]  0.0003s
walk  : [820, 524]  0.0000s


## 12. Bộ test

1. Ví dụ đề và ví dụ trường hợp (b).
2. Edge case: cạnh thừa trỏ vào gốc với chu trình dài/ngắn; đỉnh hai cha mà cạnh trên chu trình là `cand2`; chu trình độ dài 2.
3. Ngẫu nhiên có seed: so sánh mọi cách với brute force; kiểm tra thêm rằng bỏ đáp án thì thật sự được cây.

In [14]:
METHODS = [redundant_dsu, redundant_walk, lambda e: Solution().findRedundantDirectedConnection(e)]

def check(edges, expected=None):
    ref = redundant_brute(edges)
    for f in METHODS:
        got = f([e[:] for e in edges])
        assert got == ref, (edges, got, ref)
    if expected is not None:
        assert ref == expected, (edges, ref, expected)
    return ref

check(ex1, [2, 3])
check(ex2, [4, 1])
check(ex3, [2, 1])                                       # (b): cand1 trên chu trình
check([[1, 2], [2, 3], [3, 1]], [3, 1])                  # (c): chu trình độ dài 3, mọi đỉnh
check([[2, 1], [1, 2], [1, 3]], [1, 2])                  # (c): chu trình độ dài 2
check([[1, 2], [2, 3], [3, 4], [4, 2]], [4, 2])          # (b): cand2 chính là cạnh trên chu trình
check([[4, 2], [1, 5], [5, 2], [5, 3], [2, 4]], [4, 2])  # (b): cand1 = [4,2] trên chu trình 2->4->2
check([[1, 3], [2, 3], [3, 4], [1, 2]], [2, 3])          # (a) với cạnh thứ hai xuất hiện giữa danh sách
print("Ví dụ + edge case: OK")

rng = random.Random(2025)
for _ in range(2000):
    n = rng.randint(3, 9)
    e = random_case(n, rng)
    ans = check(e)
    rest = [tuple(x) for x in e if x != ans]
    assert is_rooted_tree(n, rest)
print("2000 bộ ngẫu nhiên: mọi cách khớp brute force")

Ví dụ + edge case: OK
2000 bộ ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng nguyên lời giải bài 684 (Union-Find vô hướng, trả cạnh khép chu trình)

In [15]:
def redundant_undirected(edges):
    dsu = DSU(len(edges))
    for u, v in edges:
        if not dsu.union(u, v):
            return [u, v]                 # SAI cho đồ thị có hướng
    return None

for e in [ex1, ex3]:
    print(e, "sai:", redundant_undirected(e), "| đúng:", redundant_brute(e))

[[1, 2], [1, 3], [2, 3]] sai: [2, 3] | đúng: [2, 3]
[[2, 1], [3, 1], [4, 2], [1, 4]] sai: [1, 4] | đúng: [2, 1]


**Vì sao sai:** ở trường hợp (b), cạnh khép chu trình (vô hướng) là `[1,4]`, nhưng bỏ nó thì đỉnh 1 vẫn có hai cha. Bài có hướng phải xét **đỉnh hai cha** trước. Code đúng: `redundant_dsu`.

### Lỗi 2: Có đỉnh hai cha thì luôn trả cạnh xuất hiện sau (`cand2`)

In [16]:
def redundant_always_cand2(edges):
    n = len(edges)
    parent_of = [0] * (n + 1)
    for u, v in edges:
        if parent_of[v]:
            return [u, v]                 # SAI: chưa kiểm tra chu trình
        parent_of[v] = u
    return redundant_undirected(edges)

print("ex3 sai:", redundant_always_cand2(ex3), "| đúng:", redundant_brute(ex3))

ex3 sai: [3, 1] | đúng: [2, 1]


**Vì sao sai:** nếu `cand1` nằm trên chu trình, bỏ `cand2` vẫn còn chu trình. Phải tạm bỏ `cand2` rồi kiểm tra chu trình.

### Lỗi 3: Quên trường hợp (c) — không đỉnh nào có hai cha

In [17]:
def redundant_forget_c(edges):
    n = len(edges)
    parent_of = [0] * (n + 1)
    cand1 = cand2 = None
    for u, v in edges:
        if parent_of[v]:
            cand1, cand2 = [parent_of[v], v], [u, v]
        else:
            parent_of[v] = u
    dsu = DSU(n)
    for u, v in edges:
        if [u, v] == cand2:
            continue
        if not dsu.union(u, v):
            return cand1                  # SAI: cand1 là None khi không có đỉnh hai cha
    return cand2

print("ex2 sai:", redundant_forget_c(ex2), "| đúng:", redundant_brute(ex2))

ex2 sai: None | đúng: [4, 1]


**Vì sao sai:** ví dụ 2 là trường hợp (c): `cand1 = None`. Phải trả cạnh đang khép chu trình `[u, v]`: `return cand1 if cand1 else [u, v]`.

### Lỗi 4: Ghi đè `parent_of[v]` bằng cha thứ hai

Nếu viết `parent_of[v] = u` cho cả cạnh thứ hai thì `cand1 = [parent_of[v], v]` sẽ lấy nhầm cha, và đồ thị mô tả bởi `parent_of` không còn là "đã bỏ `cand2`" — Cách 3 sẽ sai. Luôn dùng `if … else` như trong code chuẩn.

In [18]:
def redundant_overwrite(edges):
    n = len(edges)
    par = [0] * (n + 1)
    cand1 = cand2 = None
    for u, v in edges:
        if par[v]:
            cand1, cand2 = [par[v], v], [u, v]
        par[v] = u                        # SAI: ghi đè cả khi đã có cha
    if cand1:
        v, x = cand1[1], par[cand1[1]]
        for _ in range(n):
            if x == 0:
                break
            if x == v:
                return cand1
            x = par[x]
        return cand2
    return redundant_walk(edges)

bad = [e for e in (random_case(rng.randint(3, 7), rng) for _ in range(500)) if redundant_overwrite(e) != redundant_brute(e)]
print("số bộ trả lời sai trong 500:", len(bad), "| ví dụ:", bad[0] if bad else None)

số bộ trả lời sai trong 500: 77 | ví dụ: [[3, 5], [4, 2], [5, 3], [4, 3], [4, 1]]


## 14. Mẹo & nhận dạng pattern

- **"Cây có gốc + thêm một cạnh có hướng"** → nghĩ ngay tới hai triệu chứng: *đỉnh bậc vào 2* và *chu trình*.
- **"Nếu nhiều đáp án, trả cạnh xuất hiện sau cùng"** → duyệt theo thứ tự đề với DSU tự nhiên cho cạnh khép chu trình là cạnh sau cùng; hoặc duyệt `reversed(edges)`.
- **Union-Find** là công cụ chuẩn cho "phát hiện chu trình khi thêm cạnh dần". Với đồ thị có hướng, chỉ dùng được khi đã đảm bảo **mỗi đỉnh tối đa một cạnh vào**.
- **Mẹo "tạm bỏ một ứng viên rồi kiểm tra"** biến bài phân tích nhiều nhánh thành một lần duyệt.
- Đồ thị mà mỗi đỉnh có đúng một cạnh ra/vào (đồ thị hàm) → đi `n` bước chắc chắn rơi vào chu trình.

## 15. Biến thể

1. **LeetCode 684 — Redundant Connection** (vô hướng): chỉ cần DSU, trả cạnh khép chu trình.
2. **Trả về mọi cạnh có thể bỏ** (không chỉ cạnh sau cùng): (a) → cả hai ứng viên; (b) → chỉ `cand1`; (c) → mọi cạnh trên chu trình.
3. **Thêm k cạnh thừa**: không còn phân tích gọn; cần tìm cây khung có hướng (arborescence) — thuật toán Chu–Liu/Edmonds.
4. **Kiểm tra đồ thị có hướng có phải cây có gốc** → hàm `is_rooted_tree` ở mục 3.1.

Ví dụ biến thể 2:

In [19]:
def all_removable(edges):
    n = len(edges)
    return [list(e) for i, e in enumerate(edges)
            if is_rooted_tree(n, [tuple(x) for j, x in enumerate(edges) if j != i])]

for e in [ex1, ex2, ex3]:
    print(e, "-> mọi cạnh có thể bỏ:", all_removable(e))

[[1, 2], [1, 3], [2, 3]] -> mọi cạnh có thể bỏ: [[1, 3], [2, 3]]
[[1, 2], [2, 3], [3, 4], [4, 1], [1, 5]] -> mọi cạnh có thể bỏ: [[1, 2], [2, 3], [3, 4], [4, 1]]
[[2, 1], [3, 1], [4, 2], [1, 4]] -> mọi cạnh có thể bỏ: [[2, 1]]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Bỏ từng cạnh từ cuối, kiểm tra cây có gốc | `O(n²)` | `O(n)` | Trọng tài khi test; `n` nhỏ |
| 2. Trường hợp + DSU | `cand1/cand2`; DSU bỏ qua `cand2`; chu trình → `cand1` hoặc cạnh hiện tại | `O(n·α(n))` | `O(n)` | Lời giải chuẩn, gọn nhất |
| 3. Trường hợp + đi ngược cha | Đi ngược từ đỉnh hai cha; hoặc tìm chu trình duy nhất | `O(n)` | `O(n)` | Khi không muốn/không được dùng DSU; hiểu sâu cấu trúc |

## 17. Bài tập

**Bài 1 (đoán output).** `[[1,2],[2,3],[3,1],[1,4]]`, `[[3,1],[1,2],[3,2]]`, `[[2,3],[3,1],[3,4],[4,2]]`?

**Bài 2 (tính tay).** Lập bảng 2 bước (như 8.1) cho `[[5,2],[5,1],[3,1],[3,4],[3,5]]`.

**Bài 3 (điền TODO).** Hoàn thiện `find_candidates(edges)` trả về `(cand1, cand2)` hoặc `(None, None)`.

**Bài 4 (tự viết lại).** Viết lại `my_redundant(edges)` bằng DSU.

**Bài 5 (mini challenge).** Viết `root_after_removal(edges)`: trả về **gốc** của cây sau khi bỏ cạnh đáp án.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def find_candidates(edges):
    # TODO: parent_of; nếu v đã có cha -> cand1 = [parent_of[v], v], cand2 = [u, v]
    pass

# Bài 4 — TODO
def my_redundant(edges):
    # TODO: tìm cand; DSU bỏ qua cand2; chu trình -> cand1 hoặc [u, v]; hết -> cand2
    pass

# Bài 5 — TODO
def root_after_removal(edges):
    # TODO: lấy đáp án, bỏ đi, tìm đỉnh bậc vào 0
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: đếm bậc vào từng đỉnh trước.
- Mức 2: bộ 1 là (c) với chu trình 1→2→3→1; bộ 2: đỉnh 2 hai cha (1 và 3), không chu trình; bộ 3: cha của từng đỉnh là 3←2, 1←3, 4←3, 2←4 → không đỉnh nào hai cha, có chu trình 2→3→4→2.
- Mức 3: `[3,1]`, `[3,2]`, `[4,2]`.

**Bài 2.**
- Mức 1: đỉnh 1 có cha 5 rồi cha 3.
- Mức 2: `cand1=[5,1]`, `cand2=[3,1]`; DSU trên các cạnh còn lại: 5-2, 5-1, 3-4, 3-5.
- Mức 3: không chu trình → trả `cand2 = [3,1]`.

**Bài 3.**
- Mức 1: `parent_of = [0] * (n + 1)`.
- Mức 2: dùng `if parent_of[v]: ... else: parent_of[v] = u`.
- Mức 3: trả `cand1, cand2` ở cuối.

**Bài 4.**
- Mức 1: tái sử dụng `find_candidates`.
- Mức 2: `DSU(n)`; `continue` khi gặp `cand2`.
- Mức 3: `return cand1 if cand1 else [u, v]`.

**Bài 5.**
- Mức 1: gọi `Solution().findRedundantDirectedConnection`.
- Mức 2: bậc vào của các cạnh còn lại.
- Mức 3: đỉnh duy nhất có bậc vào 0.

## 19. Đáp án tham khảo

In [21]:
S = Solution()
# Bài 1
assert S.findRedundantDirectedConnection([[1, 2], [2, 3], [3, 1], [1, 4]]) == [3, 1]
assert S.findRedundantDirectedConnection([[3, 1], [1, 2], [3, 2]]) == [3, 2]
assert S.findRedundantDirectedConnection([[2, 3], [3, 1], [3, 4], [4, 2]]) == [4, 2]

# Bài 2
assert S.findRedundantDirectedConnection([[5, 2], [5, 1], [3, 1], [3, 4], [3, 5]]) == [3, 1]

# Bài 3
def find_candidates_ans(edges):
    n = len(edges)
    parent_of = [0] * (n + 1)
    cand1 = cand2 = None
    for u, v in edges:
        if parent_of[v]:
            cand1, cand2 = [parent_of[v], v], [u, v]
        else:
            parent_of[v] = u
    return cand1, cand2

assert find_candidates_ans(ex1) == ([1, 3], [2, 3])
assert find_candidates_ans(ex2) == (None, None)
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_redundant_ans(edges):
    cand1, cand2 = find_candidates_ans(edges)
    dsu = DSU(len(edges))
    for u, v in edges:
        if [u, v] == cand2:
            continue
        if not dsu.union(u, v):
            return cand1 if cand1 else [u, v]
    return cand2

# Bài 5
def root_after_removal_ans(edges):
    ans = Solution().findRedundantDirectedConnection(edges)
    n = len(edges)
    indeg = [0] * (n + 1)
    for e in edges:
        if e != ans:
            indeg[e[1]] += 1
    roots = [v for v in range(1, n + 1) if indeg[v] == 0]
    assert len(roots) == 1
    return roots[0]

assert root_after_removal_ans(ex2) == 1
assert root_after_removal_ans(ex3) == 3
rng = random.Random(7)
for _ in range(800):
    e = random_case(rng.randint(3, 8), rng)
    assert my_redundant_ans(e) == redundant_brute(e)
    r = root_after_removal_ans(e)
    rest = [tuple(x) for x in e if x != redundant_brute(e)]
    assert is_rooted_tree(len(e), rest) and all(v != r for _, v in rest)
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
parent_of = [0] * (n + 1); cand1 = cand2 = None
for u, v in edges:
    if parent_of[v]: cand1, cand2 = [parent_of[v], v], [u, v]
    else:            parent_of[v] = u
DSU; for u, v in edges:
    if [u, v] == cand2: continue
    if find(u) == find(v): return cand1 if cand1 else [u, v]
    union(u, v)
return cand2
```

**Bảng quyết định:**

| Đỉnh hai cha? | Chu trình sau khi bỏ `cand2`? | Trả về |
|---|---|---|
| có | không | `cand2` |
| có | có | `cand1` |
| không | (luôn có) | cạnh khép chu trình (sau cùng) |

**Checklist trước khi submit:**

- [ ] Tìm đỉnh hai cha **trước**, không ghi đè `parent_of[v]`.
- [ ] Bỏ qua **đúng** `cand2` khi chạy DSU.
- [ ] Chu trình + có `cand1` → trả `cand1`; chu trình + không có → trả cạnh hiện tại.
- [ ] Không chu trình → trả `cand2`.
- [ ] DSU có nén đường đi; mảng kích thước `n + 1` (đỉnh đánh số từ 1).
- [ ] Test ba trường hợp: `[[1,2],[1,3],[2,3]]`, `[[2,1],[3,1],[4,2],[1,4]]`, `[[1,2],[2,3],[3,4],[4,1],[1,5]]`.
- [ ] Tên method đúng: `findRedundantDirectedConnection(self, edges)`.